# LLM120 pretraining on Google Colab (free T4 GPU)

This notebook runs the **LLM120 from-scratch pretraining pipeline** — download → tokenizer → tokenize → **train** → generate —
on Colab's free GPU for the **300M** or **400M** model, with **checkpoints written to your Google Drive** so training survives
Colab disconnects and resumes across sessions.

### Before you start
1. Upload this entire `colab/` folder to your Google Drive so it lives at **`My Drive/colab`**.
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Open this notebook **from Drive** (in Drive, right-click the `.ipynb` → *Open with → Google Colaboratory*).

### The free-tier reality (please read)
Colab free gives a **T4 (16 GB VRAM)**, **~100 GB ephemeral disk**, **~12 GB RAM**, a **~12 h max session**, and a
**~90 min idle disconnect**. Your **free Google Drive is 15 GB** (shared with Gmail/Photos).

- A **full 10B-token run takes weeks of GPU time and a 20 GB dataset** — it does **not** fit free Drive and is not realistic on
  the free tier. This notebook therefore defaults to a **smaller, completable budget (~1B tokens)**. Raise it with the knobs below.
- The T4 has **no BF16**, so the notebook auto-selects **FP16 mixed precision** (the trainer already supports it). On an L4/A100
  it will use BF16 automatically.
- **Checkpointing is the point.** Every `SAVE_EVERY_STEPS` steps the full resumable state (model + optimizer + RNG + exact data
  position) is written to Drive. Re-running the train cell — in this session or a later one — resumes at the exact next step.

> **Scope:** this package covers **pretraining + generation** (a base completion model). The repo's later assistant stages
> (SFT / DPO / Reward / RLHF) are not wired up here — add them once the base model is coherent. See the folder's `README.md`.

In [ ]:
# ============================ EDIT THESE ============================
MODEL_SIZE         = "400m"          # "300m" or "400m"
MAX_TOKENS         = 1_000_000_000   # token budget for the whole run (1B ≈ 2 GB data, fits free Drive)
MICRO_BATCH        = 2               # T4 has 16 GB. 2 is safe; try 4 to go faster; drop to 1 if you OOM.
KEEP_LAST          = 2               # rotating checkpoints kept on Drive (each ≈ 4.8 GB for 400M). See README.
SAVE_EVERY_STEPS   = 250             # checkpoint frequency — Colab disconnects, so keep this modest
TOKENIZER_MAX_DOCS = 300_000         # BPE training documents (first-time build only)
DRIVE_PROJECT      = "/content/drive/MyDrive/colab"   # the folder you uploaded to Drive
HF_TOKEN           = ""              # optional: paste a HF token if the dataset download needs auth
# ---------------------------------------------------------------------
import math, os, subprocess, sys
assert MODEL_SIZE in {"300m", "400m"}, MODEL_SIZE
SEQ_LEN      = 2048
GLOBAL_BATCH = 65_536                # tokens per optimizer step (matches the recipe)
GRAD_ACCUM   = max(1, GLOBAL_BATCH // (SEQ_LEN * MICRO_BATCH))
CKPT_DRIVE   = f"{DRIVE_PROJECT}/checkpoints/llm120-{MODEL_SIZE}"
print(f"model={MODEL_SIZE}  max_tokens={MAX_TOKENS:,}  micro={MICRO_BATCH}  grad_accum={GRAD_ACCUM}  "
      f"global_batch={SEQ_LEN*MICRO_BATCH*GRAD_ACCUM}  ckpt={CKPT_DRIVE}")

## 1. Mount Google Drive and locate the project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir(DRIVE_PROJECT), (
    f"Not found: {DRIVE_PROJECT}. Upload the 'colab' folder to your Drive and/or fix DRIVE_PROJECT in the settings cell.")
for sub in ("artifacts", "data", "checkpoints"):
    os.makedirs(os.path.join(DRIVE_PROJECT, sub), exist_ok=True)
vfs = os.statvfs(DRIVE_PROJECT)
free_gb = vfs.f_bavail * vfs.f_frsize / 1e9
print("Project folder:", DRIVE_PROJECT)
print(f"Free space on Drive: {free_gb:.1f} GB  (free tier total = 15 GB, shared with Gmail/Photos)")
if free_gb < 6:
    print("  ⚠  Low on Drive space — lower KEEP_LAST or MAX_TOKENS, see README.")

## 2. Install dependencies
Colab already provides PyTorch with CUDA, so we install only the rest. We deliberately **skip the repo's `cu128` PyTorch index** — that index targets the author's Blackwell RTX 5050 (sm_120) and is wrong for Colab's T4 (sm_75).

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{DRIVE_PROJECT}/requirements-colab.txt"], check=True)
import torch, transformers
assert torch.cuda.is_available(), "Set Runtime → Change runtime type → T4 GPU, then re-run this cell."
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"   # T4 → fp16 ; L4 / A100 → bf16
print("GPU:", torch.cuda.get_device_name(0),
      f"| compute capability sm_{''.join(map(str, torch.cuda.get_device_capability()))}")
print("transformers:", transformers.__version__, "| torch:", torch.__version__, "| precision:", PRECISION)

In [ ]:
# Make the llm120 package (shipped in this Drive folder) importable, and define a shell helper.
PKG_SRC = os.path.join(DRIVE_PROJECT, "src")
sys.path.insert(0, PKG_SRC)
os.environ["PYTHONPATH"] = PKG_SRC
import llm120
print("llm120 import OK from", os.path.dirname(llm120.__file__))

def sh(cmd):
    # Run a command with the llm120 package on PYTHONPATH, streaming live output.
    print(f"\n$ {cmd}", flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    if HF_TOKEN:
        env["HF_TOKEN"] = HF_TOKEN
    subprocess.run(cmd, shell=True, env=env, check=True)

## 3. Prepare the data (first session only)
Downloads a subset of `EleutherAI/SmolLM2-135M-10B` sized to `MAX_TOKENS`, trains the byte-BPE tokenizer (or reuses one already on Drive), and tokenizes documents into the uint16 shard format the trainer memory-maps. The result is **cached on Drive**, so later sessions just copy it back to the VM. Training then reads from the fast local VM disk, not Drive.

*Tip:* to skip BPE training, copy your local repo's `artifacts/tokenizer-32k/` into `colab/artifacts/tokenizer-32k/` on Drive before running this — the tokenizer must match the tokenized data, so only do this before the first build.

In [ ]:
import shutil, json
LOCAL_RAW  = "/content/llm120_data/raw"
LOCAL_DATA = "/content/llm120_data/tokenized"
DATA_DRIVE = f"{DRIVE_PROJECT}/data/tokenized"
TOK_DRIVE  = f"{DRIVE_PROJECT}/artifacts/tokenizer-32k"
os.makedirs(LOCAL_RAW, exist_ok=True)

if os.path.exists(os.path.join(DATA_DRIVE, "train-manifest.json")):
    print("Tokenized corpus already on Drive → copying to VM for fast training reads...")
    if os.path.exists(LOCAL_DATA):
        shutil.rmtree(LOCAL_DATA)
    shutil.copytree(DATA_DRIVE, LOCAL_DATA)
else:
    print("First-time build: download → tokenizer → tokenize (this can take 15–40 min on a free CPU)...")
    n_shards = max(1, min(85, round(MAX_TOKENS * 1.2 / 117_000_000)))   # ~117M tokens per shard
    sh(f"python -m llm120.download --max-shards {n_shards} --output {LOCAL_RAW}")
    if not os.path.exists(os.path.join(TOK_DRIVE, "tokenizer.json")):
        sh(f"python -m llm120.tokenizer_train --input {LOCAL_RAW} --output {TOK_DRIVE} --max-documents {TOKENIZER_MAX_DOCS}")
    sh(f"python -m llm120.prepare --input {LOCAL_RAW} --tokenizer {TOK_DRIVE} --output {LOCAL_DATA} --workers 2")
    os.makedirs(os.path.dirname(DATA_DRIVE), exist_ok=True)
    shutil.copytree(LOCAL_DATA, DATA_DRIVE)   # persist on Drive so we never rebuild

manifest = json.load(open(os.path.join(LOCAL_DATA, "train-manifest.json")))
print(f"Train tokens ready: {manifest['total_tokens']:,}   (training budget cap: {MAX_TOKENS:,})")

## 4. Write the training config and run the doctor
The config is generated deterministically (sorted keys, no timestamps) so its SHA-256 is identical across sessions — that is what lets `--resume auto` accept the checkpoint next time. **Don't change any setting between sessions**, or resume will be refused as a different run.

In [ ]:
import yaml
MODELS = {
    "300m": dict(hidden_size=800, intermediate_size=2816, num_hidden_layers=33,
                 num_attention_heads=10, num_key_value_heads=2, initializer_range=0.035355339059327376),
    "400m": dict(hidden_size=960, intermediate_size=3328, num_hidden_layers=31,
                 num_attention_heads=12, num_key_value_heads=4, initializer_range=0.03227486121839514),
}[MODEL_SIZE]

config = {
    "project": {"name": f"llm120-{MODEL_SIZE}", "output_dir": CKPT_DRIVE, "seed": 42},
    "paths": {
        "tokenizer": TOK_DRIVE,
        "train_manifest": f"{LOCAL_DATA}/train-manifest.json",
        "validation_manifest": f"{LOCAL_DATA}/validation-manifest.json",
    },
    "model": {**MODELS,
              "max_position_embeddings": SEQ_LEN, "rope_theta": 10000.0,
              "rms_norm_eps": 1.0e-5, "tie_word_embeddings": True},
    "training": {
        "sequence_length": SEQ_LEN, "micro_batch_size": MICRO_BATCH,
        "gradient_accumulation_steps": GRAD_ACCUM, "max_tokens": MAX_TOKENS,
        "learning_rate": 0.0015, "warmup_steps": 2000, "decay_ratio": 0.20,
        "min_learning_rate_ratio": 0.0, "adam_beta1": 0.9, "adam_beta2": 0.95,
        "adam_epsilon": 1.0e-8, "weight_decay": 0.01, "max_grad_norm": 1.0,
        "precision": PRECISION, "gradient_checkpointing": True, "torch_compile": False,
        "allow_tf32": True, "fused_optimizer": True, "log_every_steps": 10,
        "eval_every_steps": 500, "eval_batches": 32,
        "save_every_steps": SAVE_EVERY_STEPS, "keep_last_checkpoints": KEEP_LAST},
}
CONFIG_PATH = "/content/train_colab.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f, sort_keys=True)   # sort_keys=True → stable bytes → stable SHA-256
print("Wrote", CONFIG_PATH)
sh(f"python -m llm120.doctor --config {CONFIG_PATH}")

## 5. Benchmark (~1 min) — measure real throughput
Synthetic forward/backward to measure tokens/s and VRAM on this T4. Use the printed 10B-token estimate to sanity-check your time budget (divide by your `MAX_TOKENS` for your run).

In [ ]:
sh(f"python -m llm120.benchmark --config {CONFIG_PATH} --steps 20")

## 6. Train
Runs the full budget. It **resumes automatically** from the newest checkpoint on Drive (`--resume auto`), restoring the exact optimizer step, optimizer/scaler/RNG state, and data position. **Every `SAVE_EVERY_STEPS` steps a checkpoint is written to Drive**, so a disconnect only costs the progress since the last save.

**If Colab disconnects** (12 h cap or idle timeout): in a new session, **run every cell from the top through this one** (leave the settings cell unchanged). See the full checklist at the bottom of the notebook.

In [ ]:
sh(f"python -m llm120.train --config {CONFIG_PATH} --resume auto")

## 7. Generate a sample
This is a **base completion model** — give it a document prefix, not a question.

In [ ]:
PROMPT = "The Earth revolves around"
sh(f'python -m llm120.generate --output-dir {CKPT_DRIVE} "{PROMPT}"')

## Resuming in a later session — quick checklist
Colab wipes pip installs and the VM disk on disconnect, so each new session must rebuild the environment — but **your data and checkpoints are on Drive**.

1. **Runtime → T4 GPU.**
2. **Run every cell from the top, in order**, through **§6 Train** (skip §5 Benchmark and §7 Generate). Leave the **settings** cell unchanged so the config SHA matches the checkpoint.
   - §1 re-mounts Drive, §2 reinstalls deps, the next cell re-sets the import path + `sh()`, §3 copies the cached corpus back from Drive, §4 regenerates the (identical) config, and §6 resumes at the exact next step.

### Avoid the idle disconnect
The training loop keeps the kernel busy, but the browser tab can still time out (~90 min). Keep the tab focused, or while training paste this into the browser **developer-tools console** (F12) to auto-reconnect:
```js
function ClickConnect(){ document.querySelector("colab-connect-button")?.click() }
setInterval(ClickConnect, 60000);
```

### Storage budget (free Drive = 15 GB)
Each 400M checkpoint ≈ 4.8 GB (fp32 weights + AdamW state). `KEEP_LAST=2` ≈ 9.6 GB, plus ~2 GB for a 1B-token corpus → ~12 GB, within 15 GB. For a **2–3 B** token corpus, set `KEEP_LAST=1`. The full **10 B** corpus (20 GB) does **not** fit free Drive — see the folder's `README.md` for scaling options.